# Fine-tuning BERTimbau v6 na T4 (Colab)

Driver do `train_bertimbau_v6.py` (adaptacao do trainer v4) para o pool
completo da v6 (91.080 linhas). Default do run R0: `full_iid` + DFR cell
clip 25, 2 epocas. Cada celula de treino e idempotente: o script detecta
`last/` e retoma com `--resume`; nada depende de estado so em RAM.

## 1. Ambiente

In [ ]:
!nvidia-smi
import platform
import torch
import transformers

print("python", platform.python_version())
print("torch", torch.__version__, "| transformers", transformers.__version__)
if torch.cuda.is_available():
    cap = tuple(torch.cuda.get_device_capability(0))
    name = torch.cuda.get_device_name(0)
    print("GPU", name, "capability", cap)
    if cap != (7, 5):
        print("AVISO: a spec foi dimensionada para Tesla T4 (7,5); "
              "VRAM/throughput medidos aqui podem divergir da projecao.")
    else:
        print("OK: T4 (sm_75) -> fp16 + GradScaler (bf16 nao nativo)")
else:
    print("AVISO: sem CUDA; o treino caira para CPU e nao serve para os runs cheios.")

## 2. Instalacao (nunca reinstalar torch)

In [ ]:
!pip -q install "transformers==5.17.0" scikit-learn tqdm pyarrow
import sklearn
import pyarrow
import transformers

print("transformers", transformers.__version__,
      "| sklearn", sklearn.__version__,
      "| pyarrow", pyarrow.__version__)

## 3. Drive + copia dos dados + verificacao de sha256

Copia `v6_pool.parquet`, `v6_splits.parquet`, `prepare_stats.json`,
`token_stats.json` e o script para `/content/fakenewsbr/` e confere o sha256
do pool contra o `prepare_stats.json` (aborta se divergir).

In [ ]:
import hashlib
import json
import os
import shutil

from google.colab import drive

drive.mount("/content/drive")

D = "/content/drive/MyDrive/FakenewsBR/v6"
SRC, DST = f"{D}/data", "/content/fakenewsbr"
os.makedirs(DST, exist_ok=True)
for f in ("v6_pool.parquet", "v6_splits.parquet", "prepare_stats.json",
          "token_stats.json", "train_bertimbau_v6.py"):
    shutil.copy(f"{SRC}/{f}", f"{DST}/{f}")
    print("copiado", f)


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()


stats = json.load(open(f"{DST}/prepare_stats.json", encoding="utf-8"))
want = stats["sha256"]["v6_pool.parquet"]
got = sha256(f"{DST}/v6_pool.parquet")
print("sha256 pool:", got)
assert got == want, f"sha256 divergente: esperado {want}, obtido {got}"
print("sha256 OK vs prepare_stats.json")
print("splits disponiveis:", {k: len(v) for k, v in stats["splits"].items()})

## 4. Smoke na T4 (obrigatorio antes de qualquer run cheio)

Mede tokens/s e VRAM de pico e recalcula o ETA com o valor medido.

In [ ]:
!python /content/fakenewsbr/train_bertimbau_v6.py \
  --data /content/fakenewsbr/v6_pool.parquet \
  --splits /content/fakenewsbr/v6_splits.parquet \
  --split-col full_iid --out /content/smoke --smoke --smoke-n 512 \
  --amp auto --device auto --workers 2
import json

h = json.load(open("/content/smoke/history.json", encoding="utf-8"))
tok_s = h[-1]["tokens_per_s"]
vram = h[-1]["vram_peak_gb"]
print(f"SMOKE medido: {tok_s:.0f} tokens/s pagos | VRAM pico {vram:.2f} GB")
assert vram <= 12.0, "VRAM de pico acima de 12 GB; revisar a escada de OOM"

ts = json.load(open("/content/fakenewsbr/token_stats.json", encoding="utf-8"))
cost = ts["cost_by_split"]
print(f"{'run':<26} {'tokens/epoca':>12} {'epocas':>7} {'min/epoca':>10} {'total':>8}")
for run, split, ep in (("R0 full_iid+DFR", "train_full_iid", 2),
                       ("R1 ood_wa", "train_ood_wa", 2),
                       ("Abl informative", "train_bal_iid", 2)):
    v = cost[split]["192"]["tokens_pagos_por_epoca"]
    m = v / tok_s / 60
    print(f"{run:<26} {v/1e6:>10.2f}M {ep:>7} {m:>10.1f} {ep*m:>8.1f} min")

## 5. Run default R0 (`full_iid` + DFR cell clip 25, 192, freeze 6, 2 epocas)

A celula checa se existe `last/trainer_state.json` no Drive e acrescenta
`--resume` automaticamente (idempotente).

In [ ]:
import os

D = "/content/drive/MyDrive/FakenewsBR/v6"
RUN = f"{D}/runs/full_iid_ml192_f6_on_seed42"
RESUME_FLAG = "--resume" if os.path.exists(f"{RUN}/last/trainer_state.json") else ""
print("retomando run existente no Drive" if RESUME_FLAG else "run novo")
!python /content/fakenewsbr/train_bertimbau_v6.py \
  --data /content/fakenewsbr/v6_pool.parquet \
  --splits /content/fakenewsbr/v6_splits.parquet \
  --split-col full_iid --eval-col full_iid --eval-value test \
  --out /content/runs/full_iid_ml192_f6_on_seed42 \
  --drive-out $RUN $RESUME_FLAG \
  --max-length 192 --batch-size 32 --epochs 2 --patience 1 \
  --dfr-weights cell --weight-clip 25 \
  --lr 2e-5 --freeze-layers 6 --seed 42 --amp auto --workers 2

## 5b. R0b (`full_iid` SEM DFR) - BASE x BASE com o FT v1 antigo

Mesmo split/seed/epocas do R0, sem pesos DFR, para isolar o efeito do DFR
na comparacao com o fine-tuning v1 (que nao usava pesos).

In [ ]:
import os

D = "/content/drive/MyDrive/FakenewsBR/v6"
RUN = f"{D}/runs/full_iid_ml192_f6_off_seed42"
RESUME_FLAG = "--resume" if os.path.exists(f"{RUN}/last/trainer_state.json") else ""
print("retomando run existente no Drive" if RESUME_FLAG else "run novo")
!python /content/fakenewsbr/train_bertimbau_v6.py \
  --data /content/fakenewsbr/v6_pool.parquet \
  --splits /content/fakenewsbr/v6_splits.parquet \
  --split-col full_iid --eval-col full_iid --eval-value test \
  --out /content/runs/full_iid_ml192_f6_off_seed42 \
  --drive-out $RUN $RESUME_FLAG \
  --max-length 192 --batch-size 32 --epochs 2 --patience 1 \
  --dfr-weights off \
  --lr 2e-5 --freeze-layers 6 --seed 42 --amp auto --workers 2

## 6. OOD WhatsApp R1 (`ood_wa`: teste = canal whatsapp 6.381)

Treina no pool completo sem o canal whatsapp e testa no canal retido.

In [ ]:
import os

D = "/content/drive/MyDrive/FakenewsBR/v6"
RUN = f"{D}/runs/ood_wa_ml192_f6_on_seed42_ablate-ood"
RESUME_FLAG = "--resume" if os.path.exists(f"{RUN}/last/trainer_state.json") else ""
print("retomando run existente no Drive" if RESUME_FLAG else "run novo")
!python /content/fakenewsbr/train_bertimbau_v6.py \
  --data /content/fakenewsbr/v6_pool.parquet \
  --splits /content/fakenewsbr/v6_splits.parquet \
  --ablate ood \
  --out /content/runs/ood_wa_ml192_f6_on_seed42_ablate-ood \
  --drive-out $RUN $RESUME_FLAG \
  --max-length 192 --batch-size 32 --epochs 2 --patience 1 \
  --lr 2e-5 --freeze-layers 6 --seed 42 --amp auto --workers 2

## 7. Ablacao `informative` (`bal_iid`, sem DFR)

Referencia comparavel ao A0 v4 (grupos `is_balanced_group`, 36.896 linhas).

In [ ]:
import os

D = "/content/drive/MyDrive/FakenewsBR/v6"
RUN = f"{D}/runs/bal_iid_ml192_f6_off_seed42_ablate-informative"
RESUME_FLAG = "--resume" if os.path.exists(f"{RUN}/last/trainer_state.json") else ""
print("retomando run existente no Drive" if RESUME_FLAG else "run novo")
!python /content/fakenewsbr/train_bertimbau_v6.py \
  --data /content/fakenewsbr/v6_pool.parquet \
  --splits /content/fakenewsbr/v6_splits.parquet \
  --ablate informative \
  --out /content/runs/bal_iid_ml192_f6_off_seed42_ablate-informative \
  --drive-out $RUN $RESUME_FLAG \
  --max-length 192 --batch-size 32 --epochs 2 --patience 1 \
  --lr 2e-5 --freeze-layers 6 --seed 42 --amp auto --workers 2

## 8. Coleta: resumo dos runs + copia para o Drive

In [ ]:
import glob
import json
import os
import shutil

import pandas as pd

D = "/content/drive/MyDrive/FakenewsBR/v6"
rows = []
for p in sorted(glob.glob(f"{D}/runs/*/metrics.json")):
    m = json.load(open(p, encoding="utf-8"))
    t = m.get("test") or {}
    o = m.get("ood") or {}
    rows.append({
        "run": m.get("run_id"), "n": t.get("n"), "acc": t.get("acc"),
        "macro_f1": t.get("macro_f1"),
        "worst_group_macro_f1": t.get("worst_group_macro_f1"),
        "worst_group_balanced": t.get("worst_group_balanced"),
        "ece": t.get("ece"), "ece_balanced": t.get("ece_balanced"),
        "brier": t.get("brier"), "ood_macro_f1": o.get("macro_f1"),
        "best_epoch": m.get("best_epoch"),
    })
resumo = pd.DataFrame(rows)
pd.set_option("display.width", 240)
print(resumo.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
resumo.to_csv(f"{D}/resumo.csv", index=False)

for src in glob.glob("/content/runs/*"):
    dst = f"{D}/runs/{os.path.basename(src)}"
    if os.path.isdir(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
    else:
        shutil.copy(src, dst)
print("copiado /content/runs ->", f"{D}/runs")

## 9. Download: zip de `best/` + JSON/CSVs (sem `last/`)

In [ ]:
import glob
import os
import zipfile

D = "/content/drive/MyDrive/FakenewsBR/v6"
KEEP = ["metrics.json", "calibration.json", "predictions.csv", "per_group.csv",
        "reliability.csv", "history.json", "run_config.json", "token_stats.json"]
zip_path = "/content/fakenewsbr_v6_artefatos.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for run in sorted(glob.glob(f"{D}/runs/*/")):
        rid = os.path.basename(os.path.normpath(run))
        for f in KEEP:
            p = os.path.join(run, f)
            if os.path.exists(p):
                z.write(p, f"{rid}/{f}")
        best = os.path.join(run, "best")
        if os.path.isdir(best):
            for root, _, files in os.walk(best):
                for f in files:
                    p = os.path.join(root, f)
                    z.write(p, f"{rid}/best/{os.path.relpath(p, best)}")
print("zip:", zip_path, round(os.path.getsize(zip_path) / 1e6, 1), "MB")
try:
    from google.colab import files
    files.download(zip_path)
except Exception as e:
    print("download manual:", zip_path, "|", e)